# PDF Loading for the RAG Project

Load every PDF in `data/pdf` with LangChain's `PyPDFLoader`. The loader returns one `Document` per page, including source and page metadata, ready for later chunking and indexing.

In [1]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

PROJECT_ROOT = Path.cwd()
while not ((PROJECT_ROOT / "pyproject.toml").is_file() and (PROJECT_ROOT / "data" / "pdf").is_dir()):
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Run this notebook from inside the RAG project.")
    PROJECT_ROOT = PROJECT_ROOT.parent

PDF_DIR = PROJECT_ROOT / "data" / "pdf"
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Found {len(pdf_files)} PDF file(s):")
for pdf_path in pdf_files:
    print(f"- {pdf_path.name}")

C:\Users\Ayush\AppData\Local\Temp\ipykernel_6356\214332233.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Project root: c:\Users\Ayush\OneDrive\Documents\Desktop\Autonomus\RAG
Found 2 PDF file(s):
- NIPS-2017-attention-is-all-you-need-Paper.pdf
- Report Engineering Day .pdf


## Load PDFs

Each page becomes a LangChain `Document` with source and zero-based page metadata.

In [7]:
pdf_documents = []
for pdf_path in pdf_files:
    page_documents = PyPDFLoader(str(pdf_path)).load()
    pdf_documents.extend(page_documents)
    print(f"Loaded {len(page_documents)} page(s) from {pdf_path.name}")

print(f"\nTotal pages loaded: {len(pdf_documents)}")

Loaded 11 page(s) from NIPS-2017-attention-is-all-you-need-Paper.pdf
Loaded 4 page(s) from Report Engineering Day .pdf

Total pages loaded: 15


## Preview the first page

In [8]:
first_page = pdf_documents[0]
metadata_text = repr(first_page.metadata).encode("ascii", errors="backslashreplace").decode("ascii")
print("Metadata:", metadata_text)
print("\nPage text preview:\n")
preview = first_page.page_content[:800] or "[No extractable text on this page; it may need OCR.]"
print(preview.encode("ascii", errors="backslashreplace").decode("ascii"))

Metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On English-t

## Split pages into chunks

`RecursiveCharacterTextSplitter` tries paragraph and line breaks before falling back to smaller boundaries. These settings are character counts, not token counts. Each chunk keeps its PDF source and page metadata; chunks are split within each page, so they do not cross page boundaries.

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    add_start_index=True,
)

chunks = text_splitter.split_documents(pdf_documents)

print(f"Created {len(chunks)} chunks from {len(pdf_documents)} PDF pages.")
print(f"Chunk size: {CHUNK_SIZE} characters; overlap: {CHUNK_OVERLAP} characters.")
print("\nFirst chunk metadata:", chunks[0].metadata)
print("\nFirst chunk text:\n")
print(chunks[0].page_content[:600])

Created 50 chunks from 15 PDF pages.
Chunk size: 1000 characters; overlap: 200 characters.

First chunk metadata: {'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEngl

In [7]:
import numpy as np
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
import uuid
from sklearn.metrics.pairwise import cosine_similarity
from typing import List, Dict, Any, Tuple



In [10]:

class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}")

        except Exception as e:
            print(f"Error loading embedding model: {e}")
            self.model = None

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        try:
            embeddings = self.model.encode(
                texts,
                convert_to_numpy=True
            )
            return embeddings

        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return np.array([])


embedding_manager = EmbeddingManager()

print(embedding_manager)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1260.38it/s]


Loaded embedding model: all-MiniLM-L6-v2


In [12]:
### VECTOR STORE ###

In [16]:


import os
import uuid


class VectoreStore:
    def __init__(self ,collection_name: str ="Pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path= self.persist_directory)

            self.collection_name = self.client.get_or_create_collection(name=self.collection_name, metadata={"Description": "pdf documents collection"})
            print(f"Vector store initialized at {self.persist_directory} with collection '{self.collection_name}'")
            print(f"Existing documents in the collection: {len(self.collection_name.get())}")
        
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            self.client = None



    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        if not self.client:
            print("Vector store client is not initialized.")
            return

        # prepare the data for ChromaDB
        ids = [str(uuid.uuid4()) for _ in documents]
        metadatas = [doc["metadata"] for doc in documents]
        documents_texts = [doc["text"] for doc in documents]
        embeddings_list = embeddings.tolist()

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = doc.get("metadata", {})
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.get("text", ""))
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection_name.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_texts,
                embeddings=embeddings_list
            )
            print(f"Added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection: {len(self.collection_name.get())}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")

vectorestore = VectoreStore()
vectorestore

Vector store initialized at ../data/vector_store with collection 'Collection(name=Pdf_documents)'
Existing documents in the collection: 7
